# Dunnhumby: The Complete Journey | 08 Power BI Export

Turns the clean tables and the results of notebooks 04 to 07 into the tables the Power BI dashboard is built on, and checks them. Relationships, DAX measures and report pages are built inside Power BI.

**Why a star schema:** one fact table of sales lines surrounded by small dimension tables (product, household, week, campaign) is the layout Power BI's engine is built for. Filters flow from the dimensions to the fact through one-to-many relationships, so each measure only has to be written once.

**Why SQL does the shaping and Power BI only displays:** every business rule (merchandise lines only, weeks 16 to 101, segment and lapse definitions, causal estimates) was already decided and tested in notebooks 02 to 07. Exporting finished tables means the dashboard cannot silently disagree with the analysis.

**Why Parquet:** the sales fact has about 2.4 million rows. Parquet keeps it small, keeps the data types, and Power BI Desktop reads it directly with its Parquet connector.

**No calendar dates:** the data only has day and week numbers, so the dashboard uses `week_no` on time axes instead of inventing dates. Day of week is the relative index `day % 7` from notebook 03.

**Outputs:** 10 Parquet tables in `outputs/powerbi/`.

In [1]:
!pip -q install -U duckdb
from google.colab import drive
from pathlib import Path
import duckdb, pandas as pd
drive.mount('/content/drive')
PROJ = Path('/content/drive/MyDrive/Dunnhumby Project')
DATA, OUT = PROJ/'data/processed', PROJ/'outputs/powerbi'
OUT.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
for f in sorted(DATA.glob('*.parquet')): con.execute(f"CREATE OR REPLACE VIEW {f.stem} AS SELECT * FROM '{f}'")
for t in ('campaign_table', 'coupon_redempt'): con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM '{PROJ/'data/parquet'/t}.parquet'")
q = lambda s: con.sql(s).df()
pd.set_option('display.max_columns', 50, 'display.width', 200)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 73.3 MB/s eta 0:00:00
Mounted at /content/drive


## 1. The model

| Table | Grain | Source |
|---|---|---|
| `fact_sales` | one sales line (merchandise, weeks 16 to 101) | 02 |
| `dim_product` | one product that appears in `fact_sales` | 02 |
| `dim_household` | one household, with demographics, RFM segment (04) and lapse outcome (05) | 02, 04, 05 |
| `dim_week` | one week of the analysis window, with 4-week period and half | 02 |
| `dim_campaign` | one campaign, with weeks and redemption rate | 02 |
| `bridge_household_campaign` | one targeted household and campaign, with a redeemed flag | raw campaign tables |
| `campaign_effects` | one causal estimate (07) | 07 |
| `commodity_pairs` | one commodity pair (06) | 06 |
| `retention_4w` | one 4-week follow-up period (05 definition) | 05 |
| `day_of_week` | one relative weekday index | 03 |

Discounts are stored as positive amounts (`promo_discount`, `coupon_discount`, `coupon_match_discount`), so they add up the way a reader expects. `dim_household` keeps every household: those outside the RFM or baseline groups are labelled `Not segmented` or `Not in baseline` instead of being dropped, so household counts on every page match.

In [2]:
tables = {
    'fact_sales': """
        SELECT household_key, basket_id, day, week_no, day % 7 AS day_index, trans_hour, store_id, product_id, quantity, sales_value,
               -retail_disc AS promo_discount, -coupon_disc AS coupon_discount, -coupon_match_disc AS coupon_match_discount, retail_disc < 0 AS on_promo
        FROM fact_sales
        WHERE NOT is_nonmerch AND NOT is_zero_line AND week_no BETWEEN 16 AND 101""",
    'dim_product': """
        SELECT product_id, department, commodity, sub_commodity, brand, manufacturer, size
        FROM dim_product p
        WHERE EXISTS (SELECT 1 FROM fact_sales f WHERE f.product_id = p.product_id AND NOT f.is_nonmerch AND NOT f.is_zero_line AND f.week_no BETWEEN 16 AND 101)""",
    'dim_household': """
        SELECT h.household_key, h.has_demo, h.age, h.marital_status, h.income, h.homeowner, h.hh_composition, h.hh_size, h.kids,
               h.n_campaigns, h.n_campaigns > 0 AS targeted,
               COALESCE(s.segment, 'Not segmented') AS rfm_segment, s.r AS r_score, s.f AS f_score, s.m AS m_score,
               s.spend_per_week, s.recency_days,
               COALESCE(l.frequency_tier, 'Not in baseline') AS frequency_tier, l.threshold_days, l.lapsed, l.returned, l.days_to_lapse_or_censor
        FROM dim_household h
        LEFT JOIN household_segments s ON s.household_key = h.household_key
        LEFT JOIN household_lapse l ON l.household_key = h.household_key""",
    'dim_week': """
        SELECT week_no, MIN(day) AS first_day, MAX(day) AS last_day, COUNT(*) AS days,
               (week_no - 16) // 4 + 1 AS period_4w, CASE WHEN week_no <= 58 THEN 'H1 (weeks 16-58)' ELSE 'H2 (weeks 59-101)' END AS half
        FROM dim_day
        WHERE week_no BETWEEN 16 AND 101
        GROUP BY week_no""",
    'dim_campaign': """
        SELECT campaign, type, start_day, end_day, duration_days,
               (SELECT MAX(week_no) FROM dim_day WHERE day <= c.start_day) AS start_week,
               LEAST((SELECT MAX(week_no) FROM dim_day WHERE day <= c.end_day), 101) AS end_week,
               hh_targeted, hh_redeemed, ROUND(100.0 * hh_redeemed / hh_targeted, 1) AS redemption_pct
        FROM dim_campaign c""",
    'bridge_household_campaign': """
        SELECT t.household_key, t.CAMPAIGN AS campaign,
               EXISTS (SELECT 1 FROM coupon_redempt r WHERE r.household_key = t.household_key AND r.CAMPAIGN = t.CAMPAIGN) AS redeemed
        FROM campaign_table t""",
    'campaign_effects': """
        SELECT *, CASE control_group WHEN 'other' THEN 'targeted at another time' ELSE 'never targeted' END AS control_label
        FROM campaign_effects""",
    'commodity_pairs': """
        SELECT *, dept_a <> dept_b AS cross_department, item_a || ' + ' || item_b AS pair
        FROM commodity_pairs""",
    'retention_4w': """
        WITH a AS (
          SELECT DISTINCT f.household_key, (f.week_no - 28) // 4 + 1 AS period
          FROM fact_sales f
          JOIN household_lapse l ON l.household_key = f.household_key
          WHERE NOT f.is_nonmerch AND NOT f.is_zero_line AND f.week_no BETWEEN 28 AND 99
        )
        SELECT period, 24 + 4 * period AS first_week, COUNT(*) AS active_households,
               ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM household_lapse), 1) AS active_pct
        FROM a
        GROUP BY period""",
    'day_of_week': """
        SELECT i AS day_index, 'Day ' || i AS day_label FROM range(7) t(i)"""}
for n, s in tables.items():
    con.execute(f"COPY ({s}) TO '{OUT/n}.parquet' (FORMAT PARQUET)")
    con.execute(f"CREATE OR REPLACE VIEW pbi_{n} AS SELECT * FROM '{OUT/n}.parquet'")
q(" UNION ALL ".join(f"SELECT '{n}' AS table_name, COUNT(*) AS n_rows FROM pbi_{n}" for n in tables))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table_name,n_rows
0,fact_sales,2351888
1,dim_product,86419
2,dim_household,2500
3,dim_week,86
4,dim_campaign,30
5,bridge_household_campaign,7208
6,campaign_effects,50
7,commodity_pairs,6499
8,retention_4w,18
9,day_of_week,7


## 2. Validation
The export must match the analysis exactly. Each check returns `true` when it passes:
- **Totals** reconcile with notebook 03: $6,814,347 of merchandise sales, 2,493 households and 230,892 baskets.
- **Keys** are unique in every dimension, otherwise Power BI cannot build a one-to-many relationship.
- **No orphans:** every key in the fact and bridge tables exists in its dimension, so no sales fall into a blank row in a slicer.

In [3]:
checks = {
    'sales match notebook 03': "SELECT ABS(SUM(sales_value) - 6814347) < 1 FROM pbi_fact_sales",
    'households match notebook 03': "SELECT COUNT(DISTINCT household_key) = 2493 FROM pbi_fact_sales",
    'baskets match notebook 03': "SELECT COUNT(DISTINCT basket_id) = 230892 FROM pbi_fact_sales",
    **{f'{t} key unique': f"SELECT COUNT(*) = COUNT(DISTINCT {k}) FROM pbi_{t}" for t, k in
       (('dim_product', 'product_id'), ('dim_household', 'household_key'), ('dim_week', 'week_no'), ('dim_campaign', 'campaign'), ('day_of_week', 'day_index'))},
    **{f'{a} -> {t}': f"SELECT NOT EXISTS (SELECT 1 FROM pbi_{a} x WHERE NOT EXISTS (SELECT 1 FROM pbi_{t} d WHERE d.{k} = x.{k}))" for a, t, k in
       (('fact_sales', 'dim_product', 'product_id'), ('fact_sales', 'dim_household', 'household_key'), ('fact_sales', 'dim_week', 'week_no'),
        ('fact_sales', 'day_of_week', 'day_index'), ('bridge_household_campaign', 'dim_household', 'household_key'), ('bridge_household_campaign', 'dim_campaign', 'campaign'))}}
pd.DataFrame({'passed': {c: con.sql(s).fetchone()[0] for c, s in checks.items()}})

,passed
sales match notebook 03,True
households match notebook 03,True
baskets match notebook 03,True
dim_product key unique,True
dim_household key unique,True
dim_week key unique,True
dim_campaign key unique,True
day_of_week key unique,True
fact_sales -> dim_product,True
fact_sales -> dim_household,True


A second check recomputes a few headline numbers from the exported tables. They must equal the values reported in notebooks 03, 04 and 07, and they are the values the dashboard cards must show once they are built.

In [4]:
q("""
SELECT ROUND(SUM(sales_value) / COUNT(DISTINCT household_key) / COUNT(DISTINCT week_no), 2) AS sales_per_household_per_week,
       ROUND(SUM(sales_value) / COUNT(DISTINCT basket_id), 2) AS avg_basket_value,
       (SELECT COUNT(*) FROM pbi_dim_household WHERE rfm_segment = 'Champions') AS champions,
       (SELECT ROUND(estimate, 2) FROM pbi_campaign_effects WHERE analysis = '3. DiD, matched' AND control_group = 'other') AS campaign_effect,
       (SELECT ROUND(estimate, 2) FROM pbi_campaign_effects WHERE analysis = '1. naive: campaign weeks only') AS naive_gap
FROM pbi_fact_sales
""")

,sales_per_household_per_week,avg_basket_value,champions,campaign_effect,naive_gap
0,31.78,29.51,594,1.03,31.4


## Summary
All 10 tables reconcile exactly with notebooks 03, 04 and 07, every dimension key is unique, and every fact and bridge key has a matching dimension row. The files in `outputs/powerbi/` are ready to load into Power BI Desktop (**Get data > Parquet**), where the relationships, DAX measures and report pages are built. Because the tables carry the finished business rules and results, the dashboard only displays them and cannot drift from the analysis.